# RQ7 – Practical Usefulness and Final Recommendation

**Research Question:** To what extent is the developed supervised learning solution practically useful, interpretable, and reliable for marketing decision-making?

**Dataset:** [Kaggle Link](https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import time
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

import glob, os

# Auto-detect the dataset file anywhere in the Colab/Kaggle environment
_search_paths = [
    '/kaggle/input/marketing-and-product-performance-dataset/',
    '/content/sample_data/',
    '/content/',
    '/content/drive/MyDrive/',
    '.',
]
_extensions = ['*.csv', '*.xlsx', '*.xls']
DATA_PATH = None

for _dir in _search_paths:
    for _ext in _extensions:
        _matches = glob.glob(os.path.join(_dir, '**', _ext), recursive=True) + glob.glob(os.path.join(_dir, _ext))
        _matches = [m for m in _matches if 'marketing' in m.lower() or 'product' in m.lower() or 'performance' in m.lower()]
        if _matches:
            DATA_PATH = _matches[0]
            break
    if DATA_PATH:
        break

# Final fallback: pick any CSV/Excel in /content
if DATA_PATH is None:
    for _ext in _extensions:
        _all = glob.glob(f'/content/**/{_ext}', recursive=True) + glob.glob(f'./**/{_ext}', recursive=True)
        if _all:
            DATA_PATH = _all[0]
            break

if DATA_PATH:
    print(f'Dataset found: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Dataset not found. Please upload the CSV/Excel file to Colab via:\n'
        '  Files panel (left sidebar) → Upload, then re-run this cell.')
RANDOM_STATE = 42

Dataset found: /content/marketing_and_product_performance 2.csv


In [2]:
# ── Load & Preprocess ─────────────────────────────────────────────────────────
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

candidates = [c for c in df.columns if any(k in c.lower() for k in ['revenue','sales','sale','income','profit','amount'])]
TARGET = candidates[0] if candidates else df.select_dtypes(include=np.number).var().idxmax()

df_clean = df.dropna(thresh=len(df)*0.5, axis=1).copy()
X = df_clean.drop(columns=[TARGET]).copy()
y = df_clean[TARGET]

for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_imp = SimpleImputer(strategy='median').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imp)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE)

In [3]:
# ── Multi-Criterion Evaluation ────────────────────────────────────────────────
models_dict = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(random_state=RANDOM_STATE),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':           XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, verbosity=0),
    'SVM':               SVR(kernel='rbf'),
    'k-NN':              KNeighborsRegressor(n_neighbors=5)
}

# Interpretability scores (expert-defined, 1-5 scale)
interpretability = {
    'Linear Regression': 5, 'Decision Tree': 4,
    'Random Forest': 3,     'XGBoost': 2,
    'SVM': 2,               'k-NN': 3
}

records = []
for name, model in models_dict.items():
    # Predictive performance
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)

    # Robustness via 5-fold CV
    cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2', n_jobs=-1)
    robustness = round(1 - cv_scores.std(), 4)  # Stability proxy

    records.append({
        'Model': name,
        'R2 Score': round(r2, 3),
        'Interpretability (1-5)': interpretability[name],
        'Robustness (CV stability)': robustness,
        'Train Time (s)': round(train_time, 3)
    })

decision_df = pd.DataFrame(records)
decision_df

,Model,R2 Score,Interpretability (1-5),Robustness (CV stability),Train Time (s)
0,Linear Regression,-0.003,5,0.9980,0.037
1,Decision Tree,-1.071,4,0.9472,0.321
2,Random Forest,-0.029,3,0.9926,21.683
3,XGBoost,-0.155,2,0.9900,0.673
4,SVM,-0.002,2,0.9997,3.032
5,k-NN,-0.182,3,0.9839,0.001


In [4]:
decision_df.to_csv('tables/RQ7_decision_matrix.csv', index=False)
print('Table saved.')

Table saved.


In [5]:
# ── Figure: Radar Chart ───────────────────────────────────────────────────────
from matplotlib.patches import FancyArrowPatch

# Normalise all criteria to [0, 1]
criteria = ['R2 Score', 'Interpretability (1-5)', 'Robustness (CV stability)']
df_norm = decision_df[['Model'] + criteria].copy()
for c in criteria:
    mn, mx = df_norm[c].min(), df_norm[c].max()
    df_norm[c] = (df_norm[c] - mn) / (mx - mn + 1e-9)

# Set up radar
N = len(criteria)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

palette = ['#2E4057','#048A81','#54C6EB','#F4A261','#E76F51','#6D6875']
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for i, (_, row) in enumerate(df_norm.iterrows()):
    vals = [row[c] for c in criteria]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, color=palette[i % len(palette)], label=row['Model'])
    ax.fill(angles, vals, alpha=0.08, color=palette[i % len(palette)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, fontsize=11)
ax.set_yticklabels([])
ax.set_title('Figure 7. Multi-Criterion Trade-off Analysis\n(Performance vs Interpretability vs Robustness)', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig('figures/RQ7_radar_tradeoff.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


In [6]:
# ── Final Recommendation ──────────────────────────────────────────────────────
best_r2 = decision_df.loc[decision_df['R2 Score'].idxmax(), 'Model']
best_interp = decision_df.loc[decision_df['Interpretability (1-5)'].idxmax(), 'Model']
best_balance = decision_df.loc[(decision_df['R2 Score'] * 0.5 +
                                 decision_df['Interpretability (1-5)'] / 5 * 0.25 +
                                 decision_df['Robustness (CV stability)'] * 0.25).idxmax(), 'Model']

print('=== FINAL RECOMMENDATION ===')
print(f'Best Predictive Performance : {best_r2}')
print(f'Most Interpretable          : {best_interp}')
print(f'Best Overall Balance        : {best_balance}')
print()
print('Recommendation: Use', best_balance, 'for deployment — it offers the best balance')
print('of predictive accuracy, interpretability, and robustness for marketing decisions.')

=== FINAL RECOMMENDATION ===
Best Predictive Performance : SVM
Most Interpretable          : Linear Regression
Best Overall Balance        : Linear Regression

Recommendation: Use Linear Regression for deployment — it offers the best balance
of predictive accuracy, interpretability, and robustness for marketing decisions.


## Summary

**RQ7** provides the final multi-criterion decision matrix, radar chart trade-off visualisation, and a data-driven model recommendation. Results in `tables/RQ7_decision_matrix.csv` and `figures/RQ7_radar_tradeoff.pdf`.